<a href="https://colab.research.google.com/github/janisharanjani-byte/GenAi_playground/blob/main/Customer_Support_Ticket_using_Spacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Customer Support Ticket Router

In [1]:
import json
import os
from collections import defaultdict

TEAMS = ["Billing Team", "Technical Team", "Refund Team", "Sales Team"]

EXAMPLES = {
    "Billing Team": [
        "My payment failed", "I was charged twice", "My card was declined",
        "There is an incorrect charge on my account", "My invoice amount is wrong",
    ],
    "Technical Team": [
        "Unable to login to my account", "The app keeps crashing",
        "The website is not working", "I am getting an error", "My password is not working",
    ],
    "Refund Team": [
        "I need a refund", "I want my money back", "I want to return my order",
        "Please cancel my order and refund me", "How long does a refund take",
    ],
    "Sales Team": [
        "What is the price of this product", "Do you have any discounts",
        "I want to buy a product", "Are there any offers", "Is this item in stock",
    ],
}



In [4]:
# ============================================================
#APPROACH 1: RULE-BASED (weighted phrases, lemma aware)
# ============================================================
class RuleBasedRouter:
    """Each phrase has a weight. Strong phrases ('money back') beat weak ones ('card')."""

    RULES = {
        "Billing Team": {"charged twice": 3, "payment failed": 3, "declined": 2,
                         "charge": 2, "payment": 2, "invoice": 2, "billing": 2,
                         "transaction": 2, "card": 1},
        "Technical Team": {"login": 3, "log in": 3, "password": 3, "crash": 3,
                           "error": 2, "bug": 2, "not work": 2, "not open": 2,
                           "website": 1, "app": 1},
        "Refund Team": {"refund": 4, "money back": 4, "return": 3,
                        "cancel order": 2, "send back": 2},
        "Sales Team": {"price": 3, "discount": 3, "offer": 2, "buy": 3,
                       "purchase": 2, "stock": 2, "product": 1},
    }

    def __init__(self):
        import spacy
        from spacy.matcher import PhraseMatcher

        self.nlp = spacy.load("en_core_web_sm")
        # Match on lemmas so "crashing" -> "crash", "charged" -> "charge"
        self.matcher = PhraseMatcher(self.nlp.vocab, attr="LEMMA")
        self.weights = {}
        for team, phrases in self.RULES.items():
            for phrase, weight in phrases.items():
                key = f"{team}|{phrase}"
                self.matcher.add(key, [self.nlp(phrase)])
                self.weights[key] = weight

    def route_ticket(self, ticket: str) -> dict:
        from collections import defaultdict
        doc = self.nlp(ticket.lower())
        scores = defaultdict(int)
        hits = defaultdict(list)
        for match_id, start, end in self.matcher(doc):
            key = self.nlp.vocab.strings[match_id]
            team, phrase = key.split("|")
            scores[team] += self.weights[key]
            hits[team].append(doc[start:end].text)

        if not scores:
            return {"team": "General Support", "confidence": 0.0, "details": {}}

        total = sum(scores.values())
        best = max(scores, key=scores.get)
        return {"team": best, "confidence": scores[best] / total, "details": dict(hits)}


In [7]:
# ============================================================
# APPROACH 2: SENTENCE EMBEDDINGS (semantic search)
# ============================================================
class EmbeddingRouter:
    """Embeds every example once, then compares each ticket against all of them."""

    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.35):
        from sentence_transformers import SentenceTransformer, util

        self.util = util
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold
        self.labels, sentences = [], []
        for team, examples in EXAMPLES.items():
            for sentence in examples:
                self.labels.append(team)
                sentences.append(sentence)
        self.embeddings = self.model.encode(sentences, convert_to_tensor=True)

    def route_ticket(self, ticket: str) -> dict:
        from collections import defaultdict
        query = self.model.encode(ticket, convert_to_tensor=True)
        sims = self.util.cos_sim(query, self.embeddings)[0].tolist()

        # Per team: average of its top-2 most similar examples
        per_team = defaultdict(list)
        for label, sim in zip(self.labels, sims):
            per_team[label].append(sim)
        scores = {t: sum(sorted(v, reverse=True)[:2]) / 2 for t, v in per_team.items()}

        best = max(scores, key=scores.get)
        if scores[best] < self.threshold:
            best = "General Support"
        return {"team": best, "confidence": max(scores.values()), "details": scores}


In [8]:
# ============================================================
# APPROACH 3: LLM (Claude via Anthropic API)
# ============================================================
class LLMRouter:
    """Best for long, messy or multi-issue tickets. Needs internet + API key."""

    SYSTEM = (
        "You route customer support tickets. Teams: "
        + ", ".join(TEAMS)
        + ", General Support. Reply ONLY with JSON: "
        '{"team": "<team>", "confidence": <0-1>, "reason": "<short reason>"}'
    )

    def __init__(self, model="claude-sonnet-5-5"):
        import anthropic

        self.client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
        self.model = model

    def route_ticket(self, ticket: str) -> dict:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=200,
            system=self.SYSTEM,
            messages=[{"role": "user", "content": ticket}],
        )
        text = response.content[0].text.strip().replace("```json", "").replace("```", "")
        try:
            data = json.loads(text)
        except json.JSONDecodeError:
            return {"team": "General Support", "confidence": 0.0, "details": text}
        return {
            "team": data.get("team", "General Support"),
            "confidence": float(data.get("confidence", 0)),
            "details": data.get("reason", ""),
        }

In [ ]:
# ============================================================
# RUNNER
# ============================================================
TEST_TICKETS = [
    "My payment failed while purchasing an item.",
    "I was charged twice for my order.",
    "The application keeps crashing.",
    "I want my money back.",
    "How can I return my product?",
    "Do you have any discounts available?",
    "I paid but the app froze and now I want the money returned",  # tricky, multi-issue
]


def main():
    import os
    print("Choose approach:\n  1. Rule-based\n  2. Embeddings\n  3. LLM (Claude)")
    choice = input("Enter 1, 2 or 3: ").strip()

    if choice == "1":
        router = RuleBasedRouter()
    elif choice == "2":
        router = EmbeddingRouter()
    elif choice == "3":
        if not os.getenv("ANTHROPIC_API_KEY"):
            raise SystemExit("Set the ANTHROPIC_API_KEY environment variable first.")
        router = LLMRouter()
    else:
        raise SystemExit("Invalid choice.")

    for ticket in TEST_TICKETS:
        result = router.route_ticket(ticket)
        print(f"\nTicket     : {ticket}")
        print(f"Routed to  : {result['team']}  ({result['confidence']:.0%})")
        print(f"Details    : {result['details']}")

    print("\nType your own ticket ('exit' to quit).")
    while True:
        ticket = input("\nTicket: ").strip()
        if ticket.lower() == "exit":
            break
        if ticket:
            result = router.route_ticket(ticket)
            print(f"Routed to {result['team']} ({result['confidence']:.0%}) | {result['details']}")


if __name__ == "__main__":
    main()


Choose approach:
  1. Rule-based
  2. Embeddings
  3. LLM (Claude)
Enter 1, 2 or 3: 2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Ticket     : My payment failed while purchasing an item.
Routed to  : Billing Team  (66%)
Details    : {'Billing Team': 0.6633743643760681, 'Technical Team': 0.35063357651233673, 'Refund Team': 0.44933992624282837, 'Sales Team': 0.22857874631881714}

Ticket     : I was charged twice for my order.
Routed to  : Billing Team  (71%)
Details    : {'Billing Team': 0.7100307047367096, 'Technical Team': 0.18865707516670227, 'Refund Team': 0.5052942633628845, 'Sales Team': 0.1984555944800377}

Ticket     : The application keeps crashing.
Routed to  : Technical Team  (63%)
Details    : {'Billing Team': 0.20183232426643372, 'Technical Team': 0.6333213597536087, 'Refund Team': 0.13858085125684738, 'Sales Team': 0.13038628548383713}

Ticket     : I want my money back.
Routed to  : Refund Team  (80%)
Details    : {'Billing Team': 0.4017972946166992, 'Technical Team': 0.17512352764606476, 'Refund Team': 0.7996952831745148, 'Sales Team': 0.24038547277450562}

Ticket     : How can I return my product?